# FUTURE_SYSTEM Colab Launcher

Run cells from top to bottom. This notebook mounts Google Drive, validates project/adapter files, installs dependencies, asks for `GOOGLE_API_KEY` and `HF_TOKEN` (hidden input), then launches the Gradio UI from `run_colab.py`.


In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
# Step 2: Locate FUTURE_SYSTEM directory in MyDrive
from pathlib import Path
import os

project_dir = '/content/drive/MyDrive/FUTURE_SYSTEM'
p = Path(project_dir)

if not p.exists() or not p.is_dir() or not Path(project_dir, 'run_colab.py').exists():
    raise FileNotFoundError(
        'Could not find /content/drive/MyDrive/FUTURE_SYSTEM/run_colab.py. '
        'Please add FUTURE_SYSTEM shortcut to MyDrive first, then re-run this cell.'
    )

os.chdir(project_dir)
print(f'[OK] PROJECT_DIR = {project_dir}')


In [ ]:
# Step 2.1: Validate adapter folder in FUTURE_SYSTEM
import os

adapter_candidates = [
    '/content/drive/MyDrive/FUTURE_SYSTEM/my_finetuned_llm',
    '/content/drive/MyDrive/FUTURE_SYSTEM/my_finetined_llm',
]

print('[CHECK] FUTURE_SYSTEM top-level contents:')
for name in sorted(os.listdir('/content/drive/MyDrive/FUTURE_SYSTEM')):
    tag = '<DIR>' if os.path.isdir(os.path.join('/content/drive/MyDrive/FUTURE_SYSTEM', name)) else '<FILE>'
    print(f'  {tag} {name}')

adapter_dir = next((p for p in adapter_candidates if os.path.isdir(p)), None)
if not adapter_dir:
    raise FileNotFoundError('Adapter folder missing. Expected my_finetuned_llm or my_finetined_llm under FUTURE_SYSTEM.')

print(f'\n[CHECK] ADAPTER_DIR = {adapter_dir}')
required = ['adapter_config.json', 'adapter_model.safetensors']
for f in required:
    ok = os.path.isfile(os.path.join(adapter_dir, f))
    print(f"  - {f}: {'OK' if ok else 'MISSING'}")

print('\n[CHECK] Adapter file list:')
for name in sorted(os.listdir(adapter_dir)):
    tag = '<DIR>' if os.path.isdir(os.path.join(adapter_dir, name)) else '<FILE>'
    print(f'  {tag} {name}')


In [ ]:
# Step 3: Install dependencies
!pip -q install -r requirements.txt


In [ ]:
# Step 3.1: Check GPU runtime
import torch
gpu_ok = bool(torch.cuda.is_available())
print(f"[STEP 3.1] torch.cuda.is_available() = {gpu_ok}")
if not gpu_ok:
    print("[STEP 3.1 WARN] No GPU runtime detected. Local Script Reviewer will fallback to Gemini.")
    print("[STEP 3.1 WARN] To use local adapter LLM, switch Runtime -> Change runtime type -> GPU.")


In [ ]:
# Step 4: Input GOOGLE_API_KEY (hidden)
import os, re, getpass

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter GOOGLE_API_KEY: ")

raw_key = os.getenv("GOOGLE_API_KEY", "").strip()
m = re.search(r"AIza[0-9A-Za-z_-]{20,}", raw_key)
norm_key = m.group(0) if m else raw_key
os.environ["GOOGLE_API_KEY"] = norm_key

if not norm_key:
    raise ValueError("GOOGLE_API_KEY is empty. Please enter a valid key.")

print(f"[STEP 4.1] GOOGLE_API_KEY ready: {norm_key[:8]}... (len={len(norm_key)})")


In [ ]:
# Step 5: Input HF_TOKEN (hidden)
import os, getpass

if not os.getenv("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN: ")

hf_token = os.getenv("HF_TOKEN", "").strip()
if not hf_token:
    raise ValueError("HF_TOKEN is empty. Local reviewer model loading needs HF_TOKEN.")

print(f"[STEP 5.1] HF_TOKEN ready: {hf_token[:5]}... (len={len(hf_token)})")


In [ ]:
# Step 6: Launch Gradio UI
import importlib
import gradio as gr

gr.close_all()

import run_colab
importlib.reload(run_colab)
run_colab.main(share=True)
